# 2026 CITS4012 Project
*Make sure you change the file name with your group id.*

## Group and Contributors

- Group ID: 67
- Contributors
  - Ray You (25292965) - Data Pipeline and Models Design
  - Chris Zheng ()
  - Zhihao Zhou ()

## GitHub

- https://github.com/RayYouDS/cits4012-NLP-project

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

## 0.1 Clone the Repository from GitHub

This step clones the GitHub repository, which contains all files required to run this notebook.

Do NOT run this cell in a local testing environment.

In [ ]:
import os, subprocess
from pathlib import Path

# cloning repo to colab
if not Path('/content/repo/.git').exists():
    subprocess.run([
        "git", "clone", "-q", "-b", "training-refactor",
        "https://github.com/RayYouDS/cits4012-NLP-project.git", "/content/repo"
    ], check=True)

# Set the working directory to repo root
os.chdir('/content/repo')

## 0.2 Install Required Libraries

Run the following cell to install the dependencies required to run this project in the Google Colab runtime.

In [ ]:
!pip install -r requirement.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0.3 Import Dependencies

Import most of the libraries required for the subsequent steps in advance.

In [34]:
import gc

import torch
import warnings

import random
import sentencepiece as spm
from pathlib import Path

import pandas as pd
from IPython.display import display

## 0.4 Set Global Variables

Run the following cell to configure the global variables used throughout this notebook.

In [35]:
# ================== Get Device Variable =========================
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    warnings.warn(
        "No GPU runtime detected. Falling back to CPU.",
        RuntimeWarning,
        stacklevel=2
    )

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# ============= Set PIQA Dataset Base Path ======================
BASE_PATH = Path("./datasets/PIQA")

Device: cpu
PyTorch: 2.13.0+cpu


# 1.Dataset Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Load and Clean Raw Data

The `scripts/data_loader/load_piqa_data.py` module handles data loading, conversion from JSONL to DataFrames, and training–validation splitting based on unique goals. It also applies lightweight text normalisation, including:

- Unicode normalisation: Apply NFKC normalisation to standardise compatible Unicode characters.

- Whitespace normalisation: Replace tabs and newline characters with spaces, collapse consecutive whitespace into a single space, and remove leading and trailing whitespace.

Note: PIQA contains short goal descriptions and candidate solutions, where small lexical differences can carry important semantic information. Therefore, stop words (e.g., in, with, and no), punctuation, and original casing are preserved. Stemming, lemmatisation, and other potentially meaning-altering transformations are avoided to minimise the risk of losing information relevant to physical commonsense reasoning.

In [46]:
# Demonstration
from scripts.data_loader.load_piqa_data import clean_piqa_text, piqa_to_dataframe


# Load the raw training data
raw_df = pd.read_json(
    BASE_PATH / "train.jsonl",
    lines=True,
    encoding="utf-8",
)

# Select one example for demonstration
sample = raw_df.iloc[0]

comparison = pd.DataFrame({
    "Field": ["goal", "sol1", "sol2"],
    "Original Text": [
        sample["goal"],
        sample["sol1"],
        sample["sol2"],
    ],
    "Cleaned Text": [
        clean_piqa_text(sample["goal"]),
        clean_piqa_text(sample["sol1"]),
        clean_piqa_text(sample["sol2"]),
    ],
})

display(comparison)

,Field,Original Text,Cleaned Text
0,goal,"When boiling butter, when it's ready, you can","When boiling butter, when it's ready, you can"
1,sol1,Pour it onto a plate,Pour it onto a plate
2,sol2,Pour it into a jar,Pour it into a jar


In [ ]:
# clean objects that will not be used to release RAM
del raw_df, comparison, sample

collected = gc.collect()

print(f"Garbage collected: {collected} objects")

Garbage collected: 620 objects


## 1.2 Train the Tokenizer

SentencePiece is adopted as the subword tokenisation framework because it supports language-independent tokenisation and can represent text containing diverse Unicode characters, punctuation, and different casing patterns.

The tokenizer is trained exclusively on the training set to prevent information leakage from the validation and test sets. The resulting tokenizer is then reused consistently across all datasets.

The following special tokens are used to represent sequence boundaries, padding, and unknown tokens.

| Token | ID | Description |
|---|---:|---|
| `<unk>` | 0 | Represents unknown or out-of-vocabulary tokens |
| `<s>` | 1 | Beginning-of-sequence token |
| `</s>` | 2 | End-of-sequence token |
| `<pad>` | 3 | Padding token used to align sequence lengths within a batch |
| `[CLS]` | 4 | Classification token prepended to each input sequence |
| `[SEP]` | 5 | Separator between the goal and candidate solution |

In [ ]:
# ================== Load and clean data  ===========================
train_dataset, valid_dataset, test_dataset = piqa_to_dataframe(base_path=BASE_PATH)

# ================= Prepare temporary training corpus for Tokenizer ===========================
temp_corpus = []
temp_corpus.extend(train_dataset['goal'].to_list())
temp_corpus.extend(train_dataset['sol1'].to_list())
temp_corpus.extend(train_dataset['sol2'].to_list())

random.Random(4012).shuffle(temp_corpus)

# =============== Save temp corpus ================================
output_path = Path("./tokenizer")

temp_corpus_path =  output_path / 'temp_piqa_corpus.txt'
with temp_corpus_path.open("w", encoding="utf-8") as f:
    for text in temp_corpus:
        # clean potential returns inside corpus
        text = str(text).replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
        f.write(text + "\n")


# ======================= Train Tokenizer =====================================
model_prefix = output_path / 'piqa_bpe_v3'

spm.SentencePieceTrainer.train(input=str(temp_corpus_path),
                                model_prefix=str(model_prefix),
                                model_type="bpe",

                                vocab_size=8000,
                                character_coverage=1.0,
                                byte_fallback=True,

                                unk_id=0,
                                bos_id=1,
                                eos_id=2,
                                pad_id=3,

                                user_defined_symbols=["[CLS]", "[SEP]"]
                            )

True

In [48]:
# clean objects that will not be used to release RAM
del train_dataset, valid_dataset, test_dataset, temp_corpus, output_path, temp_corpus_path, model_prefix

collected = gc.collect()

print(f"Garbage collected: {collected} objects")

Garbage collected: 0 objects


The following code validates the trained SentencePiece BPE tokenizer through three checks:

- **Subword tokenisation**: Converts a sample sentence containing punctuation, an unseen compound word, a Unicode character with an accent, and an emoji into subword pieces and their corresponding token IDs.

- **Encoding and decoding**: Decodes the generated token IDs back into text to verify that the original sentence can be reconstructed.

- **Special token verification**: Checks the IDs assigned to the unknown, beginning-of-sequence, end-of-sequence, and padding tokens, as well as the custom [CLS] and [SEP] tokens.

These checks help verify that the tokenizer can process diverse text inputs and is configured consistently with the model's input representation.

In [ ]:
# ============= Load Global Tokenizer ============================
tokenizer = spm.SentencePieceProcessor(
    model_file="./tokenizer/piqa_bpe_v3.model"
)

In [15]:
# ============== Sentence -> pieces and ids ====================
text = "An unseen-word, café, or 😊 is still tokenizable."

pieces = tokenizer.encode(text, out_type=str)
ids = tokenizer.encode(text, out_type=int)

print('Tokenized Pieces:', pieces)
print('Tokenized IDs:', ids)

Tokenized Pieces: ['▁An', '▁unse', 'en', '-', 'word', ',', '▁c', 'af', 'é', ',', '▁or', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x8A>', '▁is', '▁still', '▁to', 'k', 'en', 'iz', 'able', '.']
Tokenized IDs: [3049, 7757, 299, 7930, 5477, 7913, 269, 1421, 7974, 7913, 366, 7890, 246, 165, 158, 144, 377, 1796, 278, 7912, 299, 598, 773, 7911]


In [16]:
# ================ Sentence Restore ==========================
# restore sentence
restored = tokenizer.decode(ids)
print('Restored Sentence:', restored)

Restored Sentence: An unseen-word, café, or 😊 is still tokenizable.


In [27]:
# ================ Test Special Tokens ========================
print("UNK:", tokenizer.unk_id())  # 0
print("BOS:", tokenizer.bos_id())  # 1
print("EOS:", tokenizer.eos_id())  # 2
print("PAD:", tokenizer.pad_id())  # 3

cls_id = tokenizer.piece_to_id("[CLS]")
sep_id = tokenizer.piece_to_id("[SEP]")

print("CLS:", cls_id)
print("SEP:", sep_id)

UNK: 0
BOS: 1
EOS: 2
PAD: 3
CLS: 4
SEP: 5


In [49]:
# clean objects that will not be used to release RAM
del text, pieces, ids, restored, cls_id, sep_id

collected = gc.collect()

print(f"Garbage collected: {collected} objects")

Garbage collected: 0 objects


## 1.3 Prepare Training, Validating, Testing Dataloaders

The `scripts/data_loader/piqa_dataloader.py` module integrates the following components:

- **Tokenisation**: Converts the text fields in the DataFrames into token IDs using the trained SentencePiece tokenizer.

- **Length-based bucketing**: Uses the BucketBatchSampler class to group examples with similar sequence lengths into mini-batches, reducing padding overhead.

- **Dynamic padding and masking**: Pads sequences within each batch to a common length and generates masks to distinguish valid tokens from padding positions.

- **GPU preloading**: Provides preload_dataloader_to_gpu to preload batches onto a specified device, reducing repeated CPU-to-device data transfers when memory permits.

The following code evaluates the padding efficiency achieved by length-based bucketing.

In [ ]:
from scripts.data_loader.piqa_dataloader import get_piqa_dataloaders, preload_dataloader_to_gpu

train_loader_bucket, validate_loader_bucket, test_loader_bucket = get_piqa_dataloaders(base_path=BASE_PATH,
                                                                  tokenizer=tokenizer,
                                                                  batch_size=1000,
                                                                  use_bucket=True)

train_loader_no_bucket, validate_loader_no_bucket, test_loader_no_bucket = get_piqa_dataloaders(base_path=BASE_PATH,
                                                                             tokenizer=tokenizer,
                                                                             batch_size=1000,
                                                                             use_bucket=False)

In [24]:
total_tokens = 0
valid_tokens = 0

for input_1, input_2, mask_1, mask_2, labels in train_loader_bucket:
    total_tokens += input_1.numel() + input_2.numel()
    valid_tokens += mask_1.sum().item() + mask_2.sum().item()

padding_ratio = 1 - valid_tokens / total_tokens

print(f"Padding ratio (use bucket): {padding_ratio:.2%}")

total_tokens = 0
valid_tokens = 0

for input_1, input_2, mask_1, mask_2, labels in train_loader_no_bucket:
    total_tokens += input_1.numel() + input_2.numel()
    valid_tokens += mask_1.sum().item() + mask_2.sum().item()

padding_ratio = 1 - valid_tokens / total_tokens

print(f"Padding ratio (not use bucket): {padding_ratio:.2%}")

Padding ratio (use bucket): 34.17%
Padding ratio (not use bucket): 86.90%


Length-based bucketing substantially reduces padding overhead. In our experiment, the padding ratio decreased from 86% without bucketing to 34% with bucketing, a reduction of 52 percentage points. This improves the proportion of valid tokens processed within each mini-batch, although the actual training speedup depends on additional factors such as model architecture and hardware.

The following code tests whether tokenised sentences can be reconstructed through decoding.

In [26]:
input_1, input_2, mask_1, mask_2, labels = next(iter(train_loader_bucket))

text_1 = tokenizer.decode_ids(input_1[0][mask_1[0]].tolist())
text_2 = tokenizer.decode_ids(input_2[0][mask_2[0]].tolist())

print('Restored Text 1:', text_1)
print('Restored Text 2:', text_2)

Restored Text 1: [CLS] To prepare strawberries to be eaten, you can[SEP] Rinse the strawberries in cold water
Restored Text 2: [CLS] To prepare strawberries to be eaten, you can[SEP] Rinse the strawberries in cold milk


In [50]:
# clean objects that will not be used to release RAM
del train_loader_bucket, validate_loader_bucket, test_loader_bucket
del train_loader_no_bucket, validate_loader_no_bucket, test_loader_no_bucket
del total_tokens, valid_tokens, padding_ratio
del text_1, text_2

collected = gc.collect()

print(f"Garbage collected: {collected} objects")

Garbage collected: 7 objects


# 2. Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 2.1 Load Models

This section introduces the recurrent and Transformer-based models used in the PIQA experiments, including their architectures and implementation-specific design choices.

### Recurrent Neural Networks (BiGRU and BiLSTM)

The recurrent models use bidirectional GRU or LSTM layers to encode each goal–solution pair. A single-head self-attention module and masked mean pooling are applied to obtain a fixed-size representation for candidate scoring.

Compared with conventional recurrent text classifiers, these implementations combine bidirectional sequence modelling with an explicit self-attention mechanism to aggregate contextual token representations.

![Recurrent Encoder](./docs/figures/rnn_architecture_diagram/rnn_architecture.drawio.svg)

### Transformer Encoder

The Transformer model uses stacked encoder layers with multi-head self-attention and feed-forward networks to contextualise tokens across the input sequence. A sequence-level representation is then extracted using CLS-token pooling or masked mean pooling before candidate scoring.

Unlike the original Transformer architecture, which was introduced for sequence-to-sequence learning with separate encoder and decoder stacks, this implementation uses only the encoder stack because PIQA is formulated as a binary classification task rather than a text-generation task.

![Transformer Encoder](./docs/figures/transformer_architecture_diagram/transformer_architecture.drawio.svg)

### Shared Candidate-Scoring Framework

Both model families encode the two candidate solutions separately and apply a shared linear scoring layer to their representations. The two scores are combined into logits for binary classification, with the training objective based on cross-entropy loss.

In [51]:
# Load the models
from scripts.models.rnn_attention_refactored import RNNWithAttention
from scripts.models.transformer_bi_classifier import TransformerClassifier

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)